In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

import pandas as pd
from q6_llm_parser import parse_and_validate
from q6_lineage_scorer import compute_q6_score

PROCESSED = Path.cwd().parent / 'data' / 'processed'
Z_DIR = PROCESSED / 'z_scored'

## Load data

In [2]:
dim = pd.read_parquet(PROCESSED / 'dim_cell_lines.parquet')

print(f"Cell lines: {len(dim)}")
print(f"\ndim columns: {dim.columns.tolist()}")

Cell lines: 2220

dim columns: ['ach_id', 'cvcl_id', 'cell_line_name', 'ccle_name', 'primary_disease', 'Subtype', 'lineage', 'lineage_subtype', 'sex', 'age', 'primary_or_metastasis', 'sample_collection_site', 'growth_pattern', 'COSMICID', 'Sanger_Model_ID', 'source', 'metadata_source', 'profile_ids', 'n_profiles', 'has_real_name', 'has_full_metadata']


In [3]:
def recommend(user_query, top_n=10):
    print(f"\n{'='*60}")
    print(f"Query: {user_query}")
    print('='*60)

    # Parse LLM
    parsed = parse_and_validate(user_query, dim)
    print("\nParsed:")
    for k, v in parsed.items():
        if k != 'warnings':
            print(f"  {k}: {v}")
    if parsed.get('warnings'):
        print(f"  Warnings: {parsed['warnings']}")

    q6 = compute_q6_score(
        dim,
        query_lineage=parsed.get('lineage'),
        query_disease=parsed.get('primary_disease'),
        query_subtype=parsed.get('subtype')
    )

    top = (
        q6[q6['q6_score'] > 0]
        .sort_values('q6_score', ascending=False)
        .head(top_n)
        .reset_index(drop=True)
    )
    top.insert(0, 'rank', top.index + 1)

    print(f"\nTop {top_n}:")
    print(top.round(3).to_string(index=False))
    return top

In [4]:
result1 = recommend("EGFR in lung cancer")


Query: EGFR in lung cancer

Parsed:
  target_gene: EGFR
  lineage: lung
  primary_disease: Lung Cancer
  subtype: None
  match_level: primary_disease

Top 10:
 rank     ach_id cell_line_name lineage primary_disease  q6_score  match_level
    1 ACH-000012         HCC827    lung     Lung Cancer      0.75 same_disease
    2 ACH-000894       NCIH1869    lung     Lung Cancer      0.75 same_disease
    3 ACH-000921      NCIH157DM    lung     Lung Cancer      0.75 same_disease
    4 ACH-000916       NCIH1573    lung     Lung Cancer      0.75 same_disease
    5 ACH-000912       NCIH2286    lung     Lung Cancer      0.75 same_disease
    6 ACH-000904       NCIH2106    lung     Lung Cancer      0.75 same_disease
    7 ACH-000901        HCC1359    lung     Lung Cancer      0.75 same_disease
    8 ACH-000900         NCIH23    lung     Lung Cancer      0.75 same_disease
    9 ACH-000893       NCIH1651    lung     Lung Cancer      0.75 same_disease
   10 ACH-000015       NCIH1581    lung     Lung C

In [5]:
result2 = recommend("HER2 positive breast cancer")


Query: HER2 positive breast cancer

Parsed:
  target_gene: ERBB2
  lineage: breast
  primary_disease: Breast Cancer
  subtype: Invasive Breast Carcinoma
  match_level: subtype

Top 10:
 rank     ach_id cell_line_name lineage primary_disease  q6_score  match_level
    1 ACH-000017          SKBR3  breast   Breast Cancer      0.75 same_disease
    2 ACH-001358       MDAMB330  breast   Breast Cancer      0.75 same_disease
    3 ACH-001395        SUM44PE  breast   Breast Cancer      0.75 same_disease
    4 ACH-001394       SUM229PE  breast   Breast Cancer      0.75 same_disease
    5 ACH-001393       SUM190PT  breast   Breast Cancer      0.75 same_disease
    6 ACH-001392       SUM185PE  breast   Breast Cancer      0.75 same_disease
    7 ACH-001391       SUM159PT  breast   Breast Cancer      0.75 same_disease
    8 ACH-001390       SUM149PT  breast   Breast Cancer      0.75 same_disease
    9 ACH-001389     SUM1315MO2  breast   Breast Cancer      0.75 same_disease
   10 ACH-001388       S

In [6]:
result3 = recommend("KRAS in colon cancer")


Query: KRAS in colon cancer

Parsed:
  target_gene: KRAS
  lineage: colorectal
  primary_disease: Colon/Colorectal Cancer
  subtype: Adenocarcinoma
  match_level: primary_disease

Top 10:
 rank     ach_id cell_line_name    lineage         primary_disease  q6_score   match_level
    1 ACH-000003          CACO2 colorectal Colon/Colorectal Cancer       1.0 exact_subtype
    2 ACH-000971         HCT116 colorectal Colon/Colorectal Cancer       1.0 exact_subtype
    3 ACH-000999        SNU1040 colorectal Colon/Colorectal Cancer       1.0 exact_subtype
    4 ACH-000998            CW2 colorectal Colon/Colorectal Cancer       1.0 exact_subtype
    5 ACH-000997          HCT15 colorectal Colon/Colorectal Cancer       1.0 exact_subtype
    6 ACH-000991          SNU81 colorectal Colon/Colorectal Cancer       1.0 exact_subtype
    7 ACH-000986          HT115 colorectal Colon/Colorectal Cancer       1.0 exact_subtype
    8 ACH-000985         LS411N colorectal Colon/Colorectal Cancer       1.0 exact_

In [7]:
result4 = recommend("Any lung adenocarcinoma")


Query: Any lung adenocarcinoma

Parsed:
  target_gene: None
  lineage: lung
  primary_disease: Lung Cancer
  subtype: Non-Small Cell Lung Cancer (NSCLC), Adenocarcinoma
  match_level: subtype

Top 10:
 rank     ach_id cell_line_name lineage primary_disease  q6_score   match_level
    1 ACH-000012         HCC827    lung     Lung Cancer       1.0 exact_subtype
    2 ACH-000448       NCIH1666    lung     Lung Cancer       1.0 exact_subtype
    3 ACH-000779            PC9    lung     Lung Cancer       1.0 exact_subtype
    4 ACH-000774      RERFLCAD2    lung     Lung Cancer       1.0 exact_subtype
    5 ACH-000766       NCIH1648    lung     Lung Cancer       1.0 exact_subtype
    6 ACH-000757           A427    lung     Lung Cancer       1.0 exact_subtype
    7 ACH-000744       NCIH1623    lung     Lung Cancer       1.0 exact_subtype
    8 ACH-000733       NCIH1838    lung     Lung Cancer       1.0 exact_subtype
    9 ACH-000731        HCC2279    lung     Lung Cancer       1.0 exact_subtyp

In [8]:
recommend("SCLC")


Query: SCLC

Parsed:
  target_gene: None
  lineage: lung
  primary_disease: Lung Cancer
  subtype: Small Cell Lung Cancer (SCLC)
  match_level: subtype

Top 10:
 rank     ach_id cell_line_name lineage primary_disease  q6_score   match_level
    1 ACH-001549          LU135    lung     Lung Cancer       1.0 exact_subtype
    2 ACH-000523       NCIH1184    lung     Lung Cancer       1.0 exact_subtype
    3 ACH-001138       NCIH2141    lung     Lung Cancer       1.0 exact_subtype
    4 ACH-000610       NCIH2227    lung     Lung Cancer       1.0 exact_subtype
    5 ACH-000594         DMS153    lung     Lung Cancer       1.0 exact_subtype
    6 ACH-001364        NCIH345    lung     Lung Cancer       1.0 exact_subtype
    7 ACH-000586       NCIH1876    lung     Lung Cancer       1.0 exact_subtype
    8 ACH-001365        NCIH847    lung     Lung Cancer       1.0 exact_subtype
    9 ACH-001386        SCLC22H    lung     Lung Cancer       1.0 exact_subtype
   10 ACH-000559       NCIH1836    lun

,rank,ach_id,cell_line_name,lineage,primary_disease,q6_score,match_level
0,1,ACH-001549,LU135,lung,Lung Cancer,1.0,exact_subtype
1,2,ACH-000523,NCIH1184,lung,Lung Cancer,1.0,exact_subtype
2,3,ACH-001138,NCIH2141,lung,Lung Cancer,1.0,exact_subtype
3,4,ACH-000610,NCIH2227,lung,Lung Cancer,1.0,exact_subtype
4,5,ACH-000594,DMS153,lung,Lung Cancer,1.0,exact_subtype
5,6,ACH-001364,NCIH345,lung,Lung Cancer,1.0,exact_subtype
6,7,ACH-000586,NCIH1876,lung,Lung Cancer,1.0,exact_subtype
7,8,ACH-001365,NCIH847,lung,Lung Cancer,1.0,exact_subtype
8,9,ACH-001386,SCLC22H,lung,Lung Cancer,1.0,exact_subtype
9,10,ACH-000559,NCIH1836,lung,Lung Cancer,1.0,exact_subtype


In [9]:
recommend("LUSC")


Query: LUSC

Parsed:
  target_gene: None
  lineage: lung
  primary_disease: Lung Cancer
  subtype: Non-Small Cell Lung Cancer (NSCLC), Squamous Cell Carcinoma
  match_level: subtype

Top 10:
 rank     ach_id cell_line_name lineage primary_disease  q6_score   match_level
    1 ACH-000553            SQ1    lung     Lung Cancer       1.0 exact_subtype
    2 ACH-000878          HCC15    lung     Lung Cancer       1.0 exact_subtype
    3 ACH-000481       NCIH2170    lung     Lung Cancer       1.0 exact_subtype
    4 ACH-000737       NCIH1385    lung     Lung Cancer       1.0 exact_subtype
    5 ACH-000390         LUDLU1    lung     Lung Cancer       1.0 exact_subtype
    6 ACH-000261       RERFLCAI    lung     Lung Cancer       1.0 exact_subtype
    7 ACH-000975        HCC2450    lung     Lung Cancer       1.0 exact_subtype
    8 ACH-000442      RERFLCSQ1    lung     Lung Cancer       1.0 exact_subtype
    9 ACH-000705           LC1F    lung     Lung Cancer       1.0 exact_subtype
   10 AC

,rank,ach_id,cell_line_name,lineage,primary_disease,q6_score,match_level
0,1,ACH-000553,SQ1,lung,Lung Cancer,1.0,exact_subtype
1,2,ACH-000878,HCC15,lung,Lung Cancer,1.0,exact_subtype
2,3,ACH-000481,NCIH2170,lung,Lung Cancer,1.0,exact_subtype
3,4,ACH-000737,NCIH1385,lung,Lung Cancer,1.0,exact_subtype
4,5,ACH-000390,LUDLU1,lung,Lung Cancer,1.0,exact_subtype
5,6,ACH-000261,RERFLCAI,lung,Lung Cancer,1.0,exact_subtype
6,7,ACH-000975,HCC2450,lung,Lung Cancer,1.0,exact_subtype
7,8,ACH-000442,RERFLCSQ1,lung,Lung Cancer,1.0,exact_subtype
8,9,ACH-000705,LC1F,lung,Lung Cancer,1.0,exact_subtype
9,10,ACH-000700,NCIH2882,lung,Lung Cancer,1.0,exact_subtype


In [10]:
recommend("melanoma") 


Query: melanoma

Parsed:
  target_gene: None
  lineage: skin
  primary_disease: Skin Cancer
  subtype: Melanoma
  match_level: subtype

Top 10:
 rank     ach_id cell_line_name lineage primary_disease  q6_score   match_level
    1 ACH-000008          A101D    skin     Skin Cancer       1.0 exact_subtype
    2 ACH-002096       CP50MELB    skin     Skin Cancer       1.0 exact_subtype
    3 ACH-002040          HMVII    skin     Skin Cancer       1.0 exact_subtype
    4 ACH-002005        SKMEL19    skin     Skin Cancer       1.0 exact_subtype
    5 ACH-002004  UACC62SKINCJ1    skin     Skin Cancer       1.0 exact_subtype
    6 ACH-002003    A375SKINCJ3    skin     Skin Cancer       1.0 exact_subtype
    7 ACH-002002    A375SKINCJ2    skin     Skin Cancer       1.0 exact_subtype
    8 ACH-002001    A375SKINCJ1    skin     Skin Cancer       1.0 exact_subtype
    9 ACH-001982           NZM3    skin     Skin Cancer       1.0 exact_subtype
   10 ACH-001973          MM485    skin     Skin Cancer

,rank,ach_id,cell_line_name,lineage,primary_disease,q6_score,match_level
0,1,ACH-000008,A101D,skin,Skin Cancer,1.0,exact_subtype
1,2,ACH-002096,CP50MELB,skin,Skin Cancer,1.0,exact_subtype
2,3,ACH-002040,HMVII,skin,Skin Cancer,1.0,exact_subtype
3,4,ACH-002005,SKMEL19,skin,Skin Cancer,1.0,exact_subtype
4,5,ACH-002004,UACC62SKINCJ1,skin,Skin Cancer,1.0,exact_subtype
5,6,ACH-002003,A375SKINCJ3,skin,Skin Cancer,1.0,exact_subtype
6,7,ACH-002002,A375SKINCJ2,skin,Skin Cancer,1.0,exact_subtype
7,8,ACH-002001,A375SKINCJ1,skin,Skin Cancer,1.0,exact_subtype
8,9,ACH-001982,NZM3,skin,Skin Cancer,1.0,exact_subtype
9,10,ACH-001973,MM485,skin,Skin Cancer,1.0,exact_subtype


In [11]:
recommend("ductal breast carcinoma") 


Query: ductal breast carcinoma

Parsed:
  target_gene: None
  lineage: breast
  primary_disease: Breast Cancer
  subtype: Ductal Adenocarcinoma
  match_level: subtype

Top 10:
 rank     ach_id cell_line_name lineage primary_disease  q6_score  match_level
    1 ACH-000017          SKBR3  breast   Breast Cancer      0.75 same_disease
    2 ACH-001358       MDAMB330  breast   Breast Cancer      0.75 same_disease
    3 ACH-001395        SUM44PE  breast   Breast Cancer      0.75 same_disease
    4 ACH-001394       SUM229PE  breast   Breast Cancer      0.75 same_disease
    5 ACH-001393       SUM190PT  breast   Breast Cancer      0.75 same_disease
    6 ACH-001392       SUM185PE  breast   Breast Cancer      0.75 same_disease
    7 ACH-001391       SUM159PT  breast   Breast Cancer      0.75 same_disease
    8 ACH-001390       SUM149PT  breast   Breast Cancer      0.75 same_disease
    9 ACH-001389     SUM1315MO2  breast   Breast Cancer      0.75 same_disease
   10 ACH-001388       SUM102PT  

,rank,ach_id,cell_line_name,lineage,primary_disease,q6_score,match_level
0,1,ACH-000017,SKBR3,breast,Breast Cancer,0.75,same_disease
1,2,ACH-001358,MDAMB330,breast,Breast Cancer,0.75,same_disease
2,3,ACH-001395,SUM44PE,breast,Breast Cancer,0.75,same_disease
3,4,ACH-001394,SUM229PE,breast,Breast Cancer,0.75,same_disease
4,5,ACH-001393,SUM190PT,breast,Breast Cancer,0.75,same_disease
5,6,ACH-001392,SUM185PE,breast,Breast Cancer,0.75,same_disease
6,7,ACH-001391,SUM159PT,breast,Breast Cancer,0.75,same_disease
7,8,ACH-001390,SUM149PT,breast,Breast Cancer,0.75,same_disease
8,9,ACH-001389,SUM1315MO2,breast,Breast Cancer,0.75,same_disease
9,10,ACH-001388,SUM102PT,breast,Breast Cancer,0.75,same_disease


In [12]:
# Limitation
recommend("triple negative breast")


Query: triple negative breast

Parsed:
  target_gene: None
  lineage: breast
  primary_disease: Breast Cancer
  subtype: None
  match_level: primary_disease

Top 10:
 rank     ach_id cell_line_name lineage primary_disease  q6_score  match_level
    1 ACH-000017          SKBR3  breast   Breast Cancer      0.75 same_disease
    2 ACH-001358       MDAMB330  breast   Breast Cancer      0.75 same_disease
    3 ACH-001395        SUM44PE  breast   Breast Cancer      0.75 same_disease
    4 ACH-001394       SUM229PE  breast   Breast Cancer      0.75 same_disease
    5 ACH-001393       SUM190PT  breast   Breast Cancer      0.75 same_disease
    6 ACH-001392       SUM185PE  breast   Breast Cancer      0.75 same_disease
    7 ACH-001391       SUM159PT  breast   Breast Cancer      0.75 same_disease
    8 ACH-001390       SUM149PT  breast   Breast Cancer      0.75 same_disease
    9 ACH-001389     SUM1315MO2  breast   Breast Cancer      0.75 same_disease
   10 ACH-001388       SUM102PT  breast   B

,rank,ach_id,cell_line_name,lineage,primary_disease,q6_score,match_level
0,1,ACH-000017,SKBR3,breast,Breast Cancer,0.75,same_disease
1,2,ACH-001358,MDAMB330,breast,Breast Cancer,0.75,same_disease
2,3,ACH-001395,SUM44PE,breast,Breast Cancer,0.75,same_disease
3,4,ACH-001394,SUM229PE,breast,Breast Cancer,0.75,same_disease
4,5,ACH-001393,SUM190PT,breast,Breast Cancer,0.75,same_disease
5,6,ACH-001392,SUM185PE,breast,Breast Cancer,0.75,same_disease
6,7,ACH-001391,SUM159PT,breast,Breast Cancer,0.75,same_disease
7,8,ACH-001390,SUM149PT,breast,Breast Cancer,0.75,same_disease
8,9,ACH-001389,SUM1315MO2,breast,Breast Cancer,0.75,same_disease
9,10,ACH-001388,SUM102PT,breast,Breast Cancer,0.75,same_disease
